In [1]:
import collections
import numpy as np
import pandas as pd
import re

from argparse import Namespace

In [2]:
args = Namespace(
    raw_train_dataset_csv="data/yelp/raw_train.csv",
    raw_test_dataset_csv="data/yelp/raw_test.csv",
    proportion_subset_of_train=0.1,
    train_proportion=0.7,
    val_proportion=0.15,
    test_proportion=0.15,
    output_munged_csv="data/yelp/reviews_with_splits_lite.csv",
    seed=1337
)

In [5]:
%cd ..\\..
%pwd

D:\Source\gh-pc\PyTorchNLPBook


'D:\\Source\\gh-pc\\PyTorchNLPBook'

In [6]:
# Read raw data
train_reviews = pd.read_csv(args.raw_train_dataset_csv, header=None, names=['rating', 'review'])

### Overview of data (added)
`train_reviews` is a pandas.Dataframe with shape (560000,2). Column "rating" has type `int64`; column "review" has type `str`. Rating values: [1, 2]; there are 280000 rows for each. 

In [22]:
# ADDED
print(f"train_reviews.shape: {train_reviews.shape}")
train_reviews.info()


train_reviews.shape: (560000, 2)
<class 'pandas.DataFrame'>
RangeIndex: 560000 entries, 0 to 559999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   rating  560000 non-null  int64
 1   review  560000 non-null  str  
dtypes: int64(1), str(1)
memory usage: 8.5 MB


In [25]:
# ADDED 
train_reviews.iloc[0]

rating                                                    1
review    Unfortunately, the frustration of being Dr. Go...
Name: 0, dtype: object

In [27]:
# ADDED
type(train_reviews.iloc[0])

pandas.Series

In [28]:
# ADDED
train_reviews.iloc[0].to_dict()

{'rating': 1,
 'review': "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone answering the phone?  It's incomprehensible and not work the aggravation.  It's with regret that I feel that I have to give Dr. Goldberg 2 stars."}

In [29]:
# making the subset equal across the review classes
by_rating = collections.defaultdict(list)
for _, row in train_reviews.iterrows():
    by_rating[row.rating].append(row.to_dict())
    
review_subset = []

for _, item_list in sorted(by_rating.items()):

    n_total = len(item_list)
    n_subset = int(args.proportion_subset_of_train * n_total)
    review_subset.extend(item_list[:n_subset])

review_subset = pd.DataFrame(review_subset)

In [32]:
# ADDED
by_rating.keys()

dict_keys([1, 2])

In [35]:
# ADDED
print(type(by_rating[1]))
len(by_rating[1])

<class 'list'>


280000

In [36]:
# ADDED
by_rating[1][0]

{'rating': 1,
 'review': "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone answering the phone?  It's incomprehensible and not work the aggravation.  It's with regret that I feel that I have to give Dr. Goldberg 2 stars."}

In [38]:
# ADDED
len(review_subset)

56000

In [40]:
# ADDED
n_subset

28000

In [43]:
# ADDED
print(f"review_subset.shape: {review_subset.shape}")
review_subset.info()

review_subset.shape: (56000, 2)
<class 'pandas.DataFrame'>
RangeIndex: 56000 entries, 0 to 55999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   rating  56000 non-null  int64
 1   review  56000 non-null  str  
dtypes: int64(1), str(1)
memory usage: 875.1 KB


In [5]:
review_subset.head()

,rating,review
0,1,"Unfortunately, the frustration of being Dr. Go..."
1,1,I don't know what Dr. Goldberg was like before...
2,1,I'm writing this review to give you a heads up...
3,1,Wing sauce is like water. Pretty much a lot of...
4,1,Owning a driving range inside the city limits ...


In [6]:
train_reviews.rating.value_counts()

2    280000
1    280000
Name: rating, dtype: int64

In [7]:
review_subset.rating.value_counts()

2    28000
1    28000
Name: rating, dtype: int64

In [8]:
# Unique classes
set(review_subset.rating)

{1, 2}

#### Comment on following (added)
As above `by_rating` will be a dict with two entries (keys {1, 2}), and each entry will be a list of dicts corresponding to the DF rows (Series). 

The `for` loop with iterate over the two entries. For each, the list (DF row items) will be randomly shuffled, and then split into train, val and test sets.

In [54]:
# Splitting the subset by rating to create our new train, val, and test splits
by_rating = collections.defaultdict(list)
for _, row in review_subset.iterrows():
    by_rating[row.rating].append(row.to_dict())
    
final_list = []
np.random.seed(args.seed)

for _, item_list in sorted(by_rating.items()):

    np.random.shuffle(item_list)
    
    n_total = len(item_list)
    n_train = int(args.train_proportion * n_total)
    n_val = int(args.val_proportion * n_total)
    n_test = int(args.test_proportion * n_total)
    
    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'
    
    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'
        
    for item in item_list[n_train+n_val:n_train+n_val+n_test]:
        item['split'] = 'test'

    # Add to final list
    final_list.extend(item_list)

#### Duplication
The following block duplicates most of the preceding code block

In [55]:
# Create split data
final_list = []
np.random.seed(args.seed)

for _, item_list in sorted(by_rating.items()):

    np.random.shuffle(item_list)
    
    n_total = len(item_list)
    n_train = int(args.train_proportion * n_total)
    n_val = int(args.val_proportion * n_total)
    n_test = int(args.test_proportion * n_total)
    
    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'
    
    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'
        
    for item in item_list[n_train+n_val:n_train+n_val+n_test]:
        item['split'] = 'test'

    # Add to final list
    final_list.extend(item_list)

In [56]:
# ADDED
print(len(final_list))
print(final_list[0])
final_list[-1]

56000
{'rating': 1, 'review': "All I can say is that A) I had no other option then to go with Time Warner Cable for my internet service \\n\\nand B....\\n\\nIf you don't have a local number it will be an absolute NIGHTMARE to pay your bill! You can go through the ring of automated messages only to get to a live person (after 10-15 min) then be told they will charge you so you must go through the automated service ....(repeat ). I cringe when it comes time to pay my bill each month.\\n\\nIf you are lucky enough to avoid using them -I suggest you do so !", 'split': 'train'}


{'rating': 2,
 'review': "I've been going to Long's Garage for about 5 years now and have been very happy with their service.  They've always been honest with me.  The prices are comparable to other shops. The repairs are done quickly and professionally.  My truck is 16 years old, yet it drives like it's only a few years old thanks to Long's. \\n\\nThe only criticism I would have would be if you need something done with a truck topper.  I didn't know what to do with the rubber seal that was falling off.  Their body work person offered to epoxy it, but that didn't stay on very long.  I didn't know at the time that I could get a new seal at Truck Trends in Champaign.\\n \\nSomething that really impressed me though was how honest their detail person was about the state of the paint on my truck.  I wasn't sure how salvageable it was, and he was honest in telling me that my paint was too thin to try polish on (would rub the last of the paint off), but he did give me some recommendations on 

In [57]:
# Write split data to file
final_reviews = pd.DataFrame(final_list)

In [58]:
# ADDED
print(f"final_reviews.shape: {final_reviews.shape}")
final_reviews.info()

final_reviews.shape: (56000, 3)
<class 'pandas.DataFrame'>
RangeIndex: 56000 entries, 0 to 55999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   rating  56000 non-null  int64
 1   review  56000 non-null  str  
 2   split   56000 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.3 MB


In [12]:
final_reviews.split.value_counts()

train    39200
test      8400
val       8400
Name: split, dtype: int64

#### Comment on following (added)
Substitutions add space before basic punctuation (`[.,!?]`), then replace any "special" characters with space.

`final_reviews.review` returns the "review" column from the pandas.Dataframe (the columns is a Series object), and `.apply()` applies the specified function to each element in the Series. Then the `review` column is updated with the new data.

In [65]:
# Preprocess the reviews
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"([.,!?])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
    return text
    
final_reviews.review = final_reviews.review.apply(preprocess_text)

In [66]:
# ADDED
print(final_reviews.review.shape)
type(final_reviews.review)

(56000,)


pandas.Series

In [67]:
# ADDED
final_reviews.review.info()

<class 'pandas.Series'>
RangeIndex: 56000 entries, 0 to 55999
Series name: review
Non-Null Count  Dtype
--------------  -----
56000 non-null  str  
dtypes: str(1)
memory usage: 437.6 KB


#### Comment on the following (added)
Like the above, the elements in the `rating` column are modified with replacement values. 

`{...}.get` behaves like a reference to a `mapping.get()` method; the current value is passed as a parameter, and the mapped value is returned.

Note two means of referring to the dataframe column: `final_reviews['rating']` and 'final_reviews.rating`. The latter is a convenience provided by Pandas, but only if the name is valid as a Python identifier and don't conflice with any existing method or attribute. Referencing using the name key is generally recommended.

In [68]:
final_reviews['rating'] = final_reviews.rating.apply({1: 'negative', 2: 'positive'}.get)

In [80]:
# ADDED
final_reviews['rating'].value_counts()

rating
negative    28000
positive    28000
Name: count, dtype: int64

In [81]:
# ADDED
final_reviews['split'].value_counts()

split
train    39200
val       8400
test      8400
Name: count, dtype: int64

In [82]:
final_reviews.head()

,rating,review,split
0,negative,all i can say is that a i had no other option ...,train
1,negative,i went here once when my long time stylist mov...,train
2,negative,i don t know why i stopped here for lunch this...,train
3,negative,did i order the wrong thing ? or maybe it was ...,train
4,negative,i went here for restaurant week . the restaura...,train


#### Comment on the following (added)
This will write out the dataframe to a csv file, "data/yelp/reviews_with_splits_lite.csv". This file already existed in the data; no new file is created. Also, because the same random seed is used, the results will be the same, so nothing will have changed (except the file timestamp).

In [83]:
final_reviews.to_csv(args.output_munged_csv, index=False)